# 1. SETUP

##1.1 Mount Colab

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/567-Semester-Project'
os.makedirs(PROJECT_DIR, exist_ok=True)
%cd "$PROJECT_DIR"

Mounted at /content/drive
/content/drive/MyDrive/567-Semester-Project


##1.2 Repo Cloning

In [2]:
import os
REPO_DIR = '/content/drive/MyDrive/567-Semester-Project'

# To clone a specific branch instead of main, add the `-b <branch_name>` flag to your `git clone` command.
# Check if the repository has already been cloned by looking for the .git directory
if not os.path.exists(os.path.join(REPO_DIR, '.git')):
    print(f"Cloning repository into {REPO_DIR}...")
    os.makedirs(REPO_DIR, exist_ok=True) # Ensure the directory exists before cloning into it
    !git clone https://github.com/AbhinavSurendra/Full-Press-ML.git "$REPO_DIR"
else:
    print(f"Repository already exists at {REPO_DIR}, performing git pull...")
    %cd "$REPO_DIR"
    !git pull

%cd "$REPO_DIR"
!ls

Repository already exists at /content/drive/MyDrive/567-Semester-Project, performing git pull...
/content/drive/MyDrive/567-Semester-Project
Already up to date.
/content/drive/MyDrive/567-Semester-Project
'567 Initial Project Presentation.pdf'	 PIPELINE_OVERVIEW.md   scripts
'567 Project Proposal.pdf'		 PROJECT_PLAN.md        src
 configs				 pyproject.toml         tasks
 data					 README.md	        tests
 DATA_SCHEMA_AND_JOIN.md		 REPORT.md


## 1.2.1 Switch Branch and Pull

In [3]:
import os
REPO_DIR = '/content/drive/MyDrive/567-Semester-Project'
BRANCH = 'nic/feature-engineering'

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    print(f"Switching to branch: {BRANCH}")
    %cd "$REPO_DIR"
    !git fetch origin
    # Checkout the branch and pull latest
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
    !git branch
else:
    print("Repository not found. Please run the cloning cell above first.")

Switching to branch: nic/feature-engineering
/content/drive/MyDrive/567-Semester-Project
M	scripts/setup_venv.sh
Branch 'nic/feature-engineering' set up to track remote branch 'nic/feature-engineering' from 'origin'.
Switched to a new branch 'nic/feature-engineering'
From https://github.com/AbhinavSurendra/Full-Press-ML
 * branch            nic/feature-engineering -> FETCH_HEAD
Already up to date.
  main
* nic/feature-engineering


##1.3 Dependecy Install

In [4]:
%cd /content/drive/MyDrive/567-Semester-Project
!pip install -q -e ".[dev]"
!pip install -q py7zr

/content/drive/MyDrive/567-Semester-Project
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.3/71.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.2/494.2 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 14.7 MB/s eta 0:00:00
  Building editable for full-press-ml (pyproject.toml) ... done


##1.4 Sanity Check GPU setup

In [5]:
import sys, torch
print("python:", sys.version.split()[0])
print("torch :", torch.__version__)
print("cuda  :", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

python: 3.12.13
torch : 2.10.0+cu128
cuda  : True Tesla T4


##1.5 Download Tiny Dataset and Run Once

In [6]:
%cd /content/drive/MyDrive/567-Semester-Project
import os
RAW_DIR = 'data/raw/tiny'
if not os.path.exists(f'{RAW_DIR}/manifest.json'):
    !python scripts/download_tiny_dataset.py --config tiny --output-dir "$RAW_DIR"
else:
    print("tiny dataset already present, skipping download")

!ls -la "$RAW_DIR"

/content/drive/MyDrive/567-Semester-Project
tiny dataset already present, skipping download
total 92369
-rw------- 1 root root 94575383 Apr 18 01:55 2015-16_pbp.csv
drwx------ 2 root root     4096 Apr 18 01:55 archives
drwx------ 7 root root     4096 Apr 18 01:55 games
-rw------- 1 root root     1389 Apr 18 01:55 manifest.json


##1.6 Build the processed event/frame/possession tables

In [7]:
%cd /content/drive/MyDrive/567-Semester-Project

!python scripts/build_possessions.py \
    --games-dir data/raw/tiny/games \
    --pbp data/raw/tiny/2015-16_pbp.csv \
    --output-dir data/processed/standard

!cat data/processed/standard/audit_summary.json

/content/drive/MyDrive/567-Semester-Project
{
  "events": 2247,
  "frames": 996784,
  "games": 5,
  "invalid_frames": 3217,
  "matched_events": 2219,
  "possessions": 995,
  "unmatched_events": 28,
  "usable_label_counts": {
    "free_throws": 107,
    "made_2": 290,
    "made_3": 70,
    "missed_shot": 352,
    "turnover": 124
  },
  "usable_possessions": 943,
  "usable_split_counts": {
    "test": 198,
    "train": 563,
    "val": 182
  },
  "valid_frames": 993567
}

##1.7 stage processed CSVs to fast local disk

In [8]:
!mkdir -p /content/work/processed
!cp /content/drive/MyDrive/567-Semester-Project/data/processed/standard/*.csv /content/work/processed/
!cp /content/drive/MyDrive/567-Semester-Project/data/processed/standard/audit_summary.json /content/work/processed/
!ls -lh /content/work/processed

total 235M
-rw------- 1 root root  428 Apr 18 22:45 audit_summary.json
-rw------- 1 root root 224K Apr 18 22:45 events.csv
-rw------- 1 root root 234M Apr 18 22:45 frames.csv
-rw------- 1 root root 120K Apr 18 22:45 possessions.csv


#1.8 First Experiement XGBoost

In [9]:
%cd /content/drive/MyDrive/567-Semester-Project

!python scripts/train_baseline.py \
    --data /content/work/processed/frames.csv \
    --label-column terminal_label \
    --model xgboost \
    --aggregate-frames

/content/drive/MyDrive/567-Semester-Project
train_rows=563 eval_rows=198 eval_split=test
accuracy=0.6061
              precision    recall  f1-score   support

 free_throws       0.67      0.59      0.62        17
      made_2       0.58      0.70      0.63        60
      made_3       0.33      0.09      0.14        11
 missed_shot       0.63      0.77      0.69        79
    turnover       0.60      0.19      0.29        31

    accuracy                           0.61       198
   macro avg       0.56      0.47      0.48       198
weighted avg       0.59      0.61      0.58       198



##1.9 Second Experiment LSTM

In [ ]:
%cd /content/drive/MyDrive/567-Semester-Project
import os, torch, numpy as np, pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from torch import nn
from torch.utils.data import DataLoader

import sys
SRC = '/content/drive/MyDrive/567-Semester-Project/src'
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from full_press_ml.data.tracking_dataset import PossessionSequenceDataset
from full_press_ml.models.lstm_model import PossessionLSTM
from full_press_ml.training.train_lstm import train_epoch, evaluate

torch.manual_seed(0); np.random.seed(0)

FRAMES = '/content/work/processed/frames.csv'
CKPT_DIR = Path('/content/drive/MyDrive/567-Semester-Project/artifacts')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(FRAMES)
df = df[df.get('possession_is_usable', 1) == 1]
df = df[df['possession_id'].notna()].copy()

le = LabelEncoder()
df['label_id'] = le.fit_transform(df['terminal_label'])

drop = {'game_id','event_id','possession_id','frame_idx','possession_frame_idx',
        'split','terminal_label','pbp_join_status','label_id'}
feat_cols = [c for c in df.columns if c not in drop and pd.api.types.is_numeric_dtype(df[c])]
df[feat_cols] = df[feat_cols].apply(pd.to_numeric, errors='coerce').fillna(0.0)

train_df = df[df['split']=='train']; eval_df = df[df['split']=='test']
train_ds = PossessionSequenceDataset(train_df, feat_cols, 'label_id', max_len=100)
eval_ds  = PossessionSequenceDataset(eval_df,  feat_cols, 'label_id', max_len=100)
train_ld = DataLoader(train_ds, batch_size=64, shuffle=True)
eval_ld  = DataLoader(eval_ds,  batch_size=64)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = PossessionLSTM(len(feat_cols), 128, 2, int(df['label_id'].nunique())).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()

for epoch in range(10):
    loss = train_epoch(model, train_ld, opt, crit, device)
    print(f"epoch={epoch+1} loss={loss:.4f}")

preds, y = evaluate(model, eval_ld, device)
print("accuracy:", accuracy_score(y, preds))
print(classification_report(y, preds, target_names=le.classes_, zero_division=0))

torch.save({
    'state_dict': model.state_dict(),
    'feature_columns': feat_cols,
    'label_classes': le.classes_.tolist(),
}, CKPT_DIR / 'lstm_tiny.pt')
print("saved:", CKPT_DIR / 'lstm_tiny.pt')


##1.10 Future Sessions: JUST START FROM HERE

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
%cd /content/drive/MyDrive/567-Semester-Project
!pip install -q -e ".[dev]"
!pip install -q py7zr
!mkdir -p /content/work/processed && cp data/processed/standard/*.csv /content/work/processed/


# 0.0 Terminal Cell

In [ ]:
import sys
SRC = '/content/drive/MyDrive/567-Semester-Project/src'
if SRC not in sys.path: sys.path.insert(0, SRC)

# pull the listing via the same code the downloader uses
import importlib.util, pathlib
spec = importlib.util.spec_from_file_location(
    "dl", "/content/drive/MyDrive/567-Semester-Project/scripts/download_tiny_dataset.py"
)
dl = importlib.util.module_from_spec(spec); spec.loader.exec_module(dl)

items = dl.fetch_listing_items()
print(f"total games available upstream: {len(items)}")
print("first 3:", [i['name'] for i in items[:3]])

total games available upstream: 636
first 3: ['01.01.2016.CHA.at.TOR.7z', '01.01.2016.DAL.at.MIA.7z', '01.01.2016.NYK.at.CHI.7z']


#2.1 Improving XGBoost

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive', force_remount=False)

import os, sys
REPO_DIR   = '/content/drive/MyDrive/567-Semester-Project'
ARCHIVE_DIR = f'{REPO_DIR}/data/raw/medium/archives'   # on Drive (cached)
PBP_PATH    = f'{REPO_DIR}/data/raw/medium/2015-16_pbp.csv'
MANIFEST    = f'{REPO_DIR}/data/raw/medium/manifest.json'
LOCAL_GAMES = '/content/work/medium/games'              # local disk, ephemeral
LOCAL_PROC  = '/content/work/medium/processed'          # local disk, fast I/O for frames.csv
PROC_DIR    = f'{REPO_DIR}/data/processed/medium_standard'  # on Drive (small artifacts only)

os.makedirs(ARCHIVE_DIR, exist_ok=True)
os.makedirs(LOCAL_GAMES, exist_ok=True)
os.makedirs(LOCAL_PROC, exist_ok=True)
os.makedirs(PROC_DIR, exist_ok=True)

SRC = f'{REPO_DIR}/src'
if SRC not in sys.path: sys.path.insert(0, SRC)

os.chdir(REPO_DIR)
!pip install -q py7zr


## Download 100 archives to drive

In [11]:
import json, random, urllib.request, shutil
from pathlib import Path
import py7zr

TRACKING_INDEX_URL = (
    "https://github.com/linouk23/NBA-Player-Movements/"
    "raw/master/data/2016.NBA.Raw.SportVU.Game.Logs"
)
PBP_URL = ("https://github.com/sumitrodatta/nba-alt-awards/"
           "raw/main/Historical/PBP%20Data/2015-16_pbp.csv")

# reuse the upstream script's listing parser — deterministic with seed=9
import importlib.util
spec = importlib.util.spec_from_file_location(
    "dl", f"{REPO_DIR}/scripts/download_tiny_dataset.py")
dl = importlib.util.module_from_spec(spec); spec.loader.exec_module(dl)

items = dl.fetch_listing_items()
random.seed(9)
selected = random.sample(items, 100)
print(f"selected {len(selected)} games")

# PBP CSV (small, stays on Drive)
if not Path(PBP_PATH).exists():
    print("downloading play-by-play CSV...")
    with urllib.request.urlopen(PBP_URL) as r, open(PBP_PATH, 'wb') as f:
        shutil.copyfileobj(r, f)

# archives → Drive (cached); extracted JSON → /content (ephemeral)
manifest = []
for i, game in enumerate(selected, 1):
    name = game['name'][:-3]
    archive_path = Path(ARCHIVE_DIR) / f"{name}.7z"
    extract_dir  = Path(LOCAL_GAMES) / name

    if not archive_path.exists():
        url = f"{TRACKING_INDEX_URL}/{name}.7z"
        with urllib.request.urlopen(url) as r, open(archive_path, 'wb') as f:
            shutil.copyfileobj(r, f)

    if not extract_dir.exists():
        extract_dir.mkdir(parents=True, exist_ok=True)
        with py7zr.SevenZipFile(archive_path, mode='r') as z:
            z.extractall(path=extract_dir)

    manifest.append({'game_name': name,
                     'archive_path': str(archive_path),
                     'extract_dir': str(extract_dir)})
    if i % 10 == 0: print(f"  {i}/100 done")

with open(MANIFEST, 'w') as f:
    json.dump({'config': 'medium', 'num_games': len(manifest),
               'games': manifest, 'pbp_path': PBP_PATH}, f, indent=2)

!du -sh "$ARCHIVE_DIR" "$LOCAL_GAMES"

selected 100 games
downloading play-by-play CSV...
  10/100 done
  20/100 done
  30/100 done
  40/100 done
  50/100 done
  60/100 done
  70/100 done
  80/100 done
  90/100 done
  100/100 done
557M	/content/drive/MyDrive/567-Semester-Project/data/raw/medium/archives
9.3G	/content/work/medium/games


##build processed tables

In [ ]:
from pathlib import Path

if not Path(f'{LOCAL_PROC}/frames.csv').exists():
    !python scripts/build_possessions.py \
        --games-dir "$LOCAL_GAMES" \
        --pbp "$PBP_PATH" \
        --output-dir "$LOCAL_PROC"
else:
    print(f"{LOCAL_PROC}/frames.csv already built, skipping")

# mirror small artifacts back to Drive for persistence (frames.csv stays local)
!cp "$LOCAL_PROC"/possessions.csv "$LOCAL_PROC"/events.csv "$LOCAL_PROC"/audit_summary.json "$PROC_DIR"/ 2>/dev/null || true

!cat "$LOCAL_PROC/audit_summary.json"
!du -sh "$LOCAL_PROC"/*


##Extract Posession Level Features

In [ ]:
import pandas as pd
from full_press_ml.features.engineer import build_frame_aggregate_table

FEATS_PATH = f'{PROC_DIR}/possession_features.csv'

if not os.path.exists(FEATS_PATH):
    # dtype hints keep memory manageable when reading the big frames table
    numeric32 = ['game_clock','shot_clock','ball_x','ball_y','ball_z',
                 'offense_centroid_x','offense_centroid_y',
                 'offense_mean_radius','offense_mean_distance_to_ball',
                 'defense_centroid_x','defense_centroid_y',
                 'defense_mean_radius','defense_mean_distance_to_ball']
    dtype_hint = {c: 'float32' for c in numeric32}
    dtype_hint.update({'is_valid_frame':'int8', 'missing_shot_clock':'int8',
                       'player_count':'int8', 'offense_player_count':'int8',
                       'defense_player_count':'int8'})

    print("reading frames.csv ...")
    frames = pd.read_csv(f'{LOCAL_PROC}/frames.csv', dtype=dtype_hint,
                         low_memory=False)
    frames = frames[frames['possession_id'].notna()]
    if 'possession_is_usable' in frames.columns:
        frames = frames[frames['possession_is_usable'] == 1]
    print(f"  {len(frames):,} usable frame rows")

    print("aggregating ...")
    agg = build_frame_aggregate_table(frames)
    agg.to_csv(FEATS_PATH, index=False)
    del frames
    print(f"saved {FEATS_PATH} with {len(agg):,} possessions")
else:
    print(f"{FEATS_PATH} already built")

!du -sh "$FEATS_PATH"


## Train XGBoost on 100 games

In [ ]:
import pandas as pd, numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from full_press_ml.models.baselines import build_xgboost

agg = pd.read_csv(f'{PROC_DIR}/possession_features.csv')
agg = agg[agg['terminal_label'].notna()].copy()

train_df = agg[agg['split'].isin(['train', 'val'])].copy()
test_df  = agg[agg['split'] == 'test'].copy()
print(f"train: {len(train_df):,}  test: {len(test_df):,}")
print("test class balance:", test_df['terminal_label'].value_counts().to_dict())

drop = {'game_id','possession_id','split','event_ids','end_reason',
        'is_usable','possession_is_usable','terminal_label'}
feat_cols = [c for c in train_df.columns if c not in drop]

X_train = train_df[feat_cols].apply(pd.to_numeric, errors='coerce').fillna(0.0)
X_test  = test_df[feat_cols].apply(pd.to_numeric, errors='coerce').fillna(0.0)
le = LabelEncoder()
y_train = le.fit_transform(train_df['terminal_label'])
y_test  = le.transform(test_df['terminal_label'])

model = build_xgboost(num_classes=len(le.classes_))
sw = compute_sample_weight('balanced', y_train)   # fights majority-class bias
model.fit(X_train, y_train, sample_weight=sw)
pred = model.predict(X_test)

print(f"\naccuracy = {accuracy_score(y_test, pred):.4f}")
print(classification_report(y_test, pred, target_names=le.classes_, zero_division=0))
print("confusion matrix (rows=true, cols=pred):")
print(pd.DataFrame(confusion_matrix(y_test, pred),
                   index=le.classes_, columns=le.classes_))
from google.colab import output
output.eval_js('new Audio("https://upload.wikimedia.org/wikipedia/commons/0/05/Beep-09.ogg").play()')
